<a href="https://colab.research.google.com/github/esther033/datascience-lab/blob/main/week3/3_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Apriori & FP tree

In [ ]:
!pip install pip --upgrade

In [ ]:
!pip install mlxtend==0.18

In [ ]:
import mlxtend
mlxtend.__version__

'0.18.0'

In [ ]:
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [ ]:
import pandas as pd
import os

## Dataset import 및 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

basicpath = '/content/drive/MyDrive/Colab Notebooks/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = basicpath + 'DS/2-3주차'
file = 'BreadBasket_DMS.csv'
data = pd.read_csv(os.path.join(path, file), index_col = None)

In [ ]:
data.head(10)

,Date,Time,Transaction,Item
0,2016.10.30,9:58:11,1,Bread
1,2016.10.30,10:05:34,2,Scandinavian
2,2016.10.30,10:05:34,2,Scandinavian
3,2016.10.30,10:07:57,3,Hot chocolate
4,2016.10.30,10:07:57,3,Jam
5,2016.10.30,10:07:57,3,Cookies
6,2016.10.30,10:08:41,4,Muffin
7,2016.10.30,10:13:03,5,Coffee
8,2016.10.30,10:13:03,5,Pastry
9,2016.10.30,10:13:03,5,Bread


In [ ]:
data.shape

(21293, 4)

## 과제1-1 Data Cleaning


위 데이터를 가공하여, transaction과 item이 column이 되는  
다음과 같은 형태의 새로운 DataFrame을 만들어 변수 data에 저장하시오.

 transaction | item
:-------------:|:------:
      0      |   7  
      0      |   14
      1      |   9  
      2      |   18
...  


In [ ]:
# 1. 대문자 -> 소문자 변환 (str.lower() 활용)
def bread(x):
    if isinstance(x, str):
        return x.lower().strip()
    return x

data['Item'] = data['Item'].apply(bread)

In [ ]:
# 2. None value 제거 (drop 활용)
data = data.dropna(subset=['Item'])

In [ ]:
data.head(10)

,Date,Time,Transaction,Item
0,2016.10.30,9:58:11,1,bread
1,2016.10.30,10:05:34,2,scandinavian
2,2016.10.30,10:05:34,2,scandinavian
3,2016.10.30,10:07:57,3,hot chocolate
4,2016.10.30,10:07:57,3,jam
5,2016.10.30,10:07:57,3,cookies
6,2016.10.30,10:08:41,4,muffin
7,2016.10.30,10:13:03,5,coffee
8,2016.10.30,10:13:03,5,pastry
9,2016.10.30,10:13:03,5,bread


### Data 간단 분석_item당 등장횟수

In [ ]:
data

,Date,Time,Transaction,Item
0,2016.10.30,9:58:11,1,bread
1,2016.10.30,10:05:34,2,scandinavian
2,2016.10.30,10:05:34,2,scandinavian
3,2016.10.30,10:07:57,3,hot chocolate
4,2016.10.30,10:07:57,3,jam
...,...,...,...,...
21288,2017.4.9,14:32:58,9682,coffee
21289,2017.4.9,14:32:58,9682,tea
21290,2017.4.9,14:57:06,9683,coffee
21291,2017.4.9,14:57:06,9683,pastry


In [ ]:
len(data['Item'].unique())

94

In [ ]:
top10_items = data['Item'].value_counts().head(10)
top10_items

,count
Item,
coffee,5471
bread,3325
tea,1435
cake,1025
pastry,856
sandwich,771
medialuna,616
hot chocolate,590
cookies,540


In [ ]:
len(data['Transaction'].unique())

9465

## 과제 1-2 one-hot-encoding vector 만들기
위 data를 가공하여 one-hot-encoding 형태의 데이터를 만들고  
이를 hot_encoded_data 변수에 저장하시오  
이때 one-hot-encoding 형태의 column은 item, row는 transaction이어야 함  

In [ ]:
#답변
hot_encoded_data = pd.crosstab(data['Transaction'], data['Item'])

In [ ]:
hot_encoded_data.head(5000)

Item,adjustment,afternoon with the baker,alfajores,argentina night,art tray,bacon,baguette,bakewell,bare popcorn,basket,...,the bart,the nomad,tiffin,toast,truffles,tshirt,valentine's card,vegan feast,vegan mincepie,victorian sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5136,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5137,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
5138,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
#아래의 코드를 설명하시오:
def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1

hot_encoded_data = hot_encoded_data.applymap(encode_units)
hot_encoded_data
# 각 값이 1 이상이면 1, 0 이하이면 0으로 변환하여 item의 존재 여부를 나타내는 이진화 코드이며,
# 하나의 transaction에 동일한 item이 여러 번 포함된 경우에도 1로 처리

/tmp/ipykernel_5740/316680575.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  hot_encoded_data = hot_encoded_data.applymap(encode_units)


Item,adjustment,afternoon with the baker,alfajores,argentina night,art tray,bacon,baguette,bakewell,bare popcorn,basket,...,the bart,the nomad,tiffin,toast,truffles,tshirt,valentine's card,vegan feast,vegan mincepie,victorian sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9680,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9681,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
9682,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 과제 1-3
mlxtend의 apriori & fp-growth 라이브러리를 활용하여 위 데이터에서 frequent itemset을 추출하시오.  
이때, min_support는 0.02로 하시오.  
association_rules 라이브러리를 활용하여 추출한 frequent itemset에서 association rule을 만드시오.

In [ ]:
#답변
from mlxtend.frequent_patterns import apriori

frequent_itemset = apriori(
    hot_encoded_data,
    min_support=0.02,
    use_colnames=True
)

frequent_itemset

,support,itemsets
0,0.036344,(alfajores)
1,0.327205,(bread)
2,0.040042,(brownie)
3,0.103856,(cake)
4,0.478394,(coffee)
5,0.054411,(cookies)
6,0.039197,(farm house)
7,0.058320,(hot chocolate)
8,0.038563,(juice)
9,0.061807,(medialuna)


In [ ]:
from mlxtend.frequent_patterns import fpgrowth

frequent_itemset = fpgrowth(
    hot_encoded_data,
    min_support=0.02,
    use_colnames=True
)

frequent_itemset

,support,itemsets
0,0.327205,(bread)
1,0.029054,(scandinavian)
2,0.058320,(hot chocolate)
3,0.054411,(cookies)
4,0.038457,(muffin)
5,0.478394,(coffee)
6,0.086107,(pastry)
7,0.061807,(medialuna)
8,0.142631,(tea)
9,0.039197,(farm house)


In [ ]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemset,
    metric='lift',
    min_threshold=1.2
)

rules[rules['confidence']> 0.14]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
0,(cake),(tea),0.103856,0.142631,0.023772,0.228891,1.604781,0.008959,1.111865
1,(tea),(cake),0.142631,0.103856,0.023772,0.166667,1.604781,0.008959,1.075372
3,(toast),(coffee),0.033597,0.478394,0.023666,0.704403,1.472431,0.007593,1.764582
